# Local inner-loop adapter: guided implementation

This notebook is the implementation workbench for the first real Carrot Tuning component. We prototype one adapter function here, inspect and refine it, then move its reviewed version into the actual adapter source. It does not replace the runner with notebook-only logic.

We will work one reviewed step at a time: define and exercise a small prototype here, inspect its useful facts once they exist, and then decide what belongs in the reusable Python adapter.

## Boundaries for this workbench

- The adapter is local only and calls the proven parent evaluator only after validation.
- The historical record is a reproduction oracle, not automatic permission to run it.
- Candidate values, source-policy changes, Foundry, Azure, and external services remain outside this notebook unless separately approved.
- A notebook cell may prototype and verify one adapter function, but only the reviewed reusable behavior belongs in the Python files under evaluator/.

## Adapter implementation checklist

- [x] Locate the existing historical contract for notebook orientation.
- [x] Prototype `read_spec` and inspect its successful read.
- [x] Prototype `validate_spec_schema`, including one malformed-contract example.
- [x] Prototype explicit target binding and fingerprint matching.
- [x] Prototype lifecycle authorization.
- [x] Validate declared local controls and the one-field candidate envelope.
- [x] Plan deterministic baseline-cache identities and locations (without running games).
- [x] Inspect planned cache paths before materialization (without running games).
- [x] Validate an existing cache payload before reuse (without running games).
- [x] Read a present gzip cache into the validator (without running games).
- [x] Decide cache reuse versus later materialization (without running games).
- [x] Define one selected missing baseline materializer (not yet invoked).
- [x] Create and validate the approved smallest-matched smoke-test contract (not yet run).
- [x] Execute and reopen both approved smoke-test baseline caches after the simulator prerequisite is available.
- [x] Promote declared-candidate selection and candidate evaluation/audit wrappers into the source runner.
- [x] Promote the reviewed local smoke evidence-packet model and fail-closed writer.
- [ ] Orchestrate candidate evaluation, evidence writing, and historical reproduction through `run()`.

A checked item means the notebook prototype was reviewed and its reusable function has been moved into `evaluator/runner.py`; it does not mean an experiment has been authorized or run.

## 9.1 Locate the existing historical experiment contract

The source-controlled historical contract and its schema already exist. This setup cell only finds their files, loads the historical JSON for notebook orientation, and makes the real adapter package importable. It establishes that the files are present and the specification parses as JSON; it does not use the reusable `read_spec` function, validate the schema, or execute a simulator.

In [181]:
import json
import sys
from pathlib import Path


def find_workspace_file(relative_path: Path) -> Path:
    """Find a checked-in project file when the notebook server chooses a different cwd."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / relative_path
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Could not find workspace file: {relative_path}")


# The adapter owns reusable behavior, so this notebook imports it rather than copying it.
adapter_source = find_workspace_file(Path("carrot-tuning-system/evaluator/runner.py"))
project_root = adapter_source.parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# This existing historical record is loaded only for notebook orientation.
# Its presence and JSON parsing are not schema validation.
spec_path = find_workspace_file(
    Path("carrot-tuning-system/experiments/historical/carrot_investment/seed_opportunity_coarse_v1.json")
)
schema_path = find_workspace_file(
    Path("carrot-tuning-system/experiments/schema/experiment-spec-v1.schema.json")
)
historical_spec = json.loads(spec_path.read_text())

# The next cell presents loaded facts; it performs no schema validation or evaluator call.

In [182]:
from IPython.display import Markdown, display


def render_contract_checkpoint(spec: dict, source: Path) -> None:
    """Presentation only: clearing this output never changes the loaded contract."""
    parameter = spec["parameter"]
    candidates = ", ".join(str(value) for value in parameter["candidate_values"])
    display(Markdown(
        f"""### Contract loaded for implementation work

| Item | Checked-in source |
|:---|:---|
| Historical specification | {spec['experiment_id']} |
| Schema version | {spec['schema_version']} |
| Lifecycle | {spec['lifecycle_status']} |
| Parameter envelope | {parameter['name']}: baseline {parameter['baseline_value']:g}; candidates {candidates} |
| Adapter source | {source.parent.name}/{source.name} |

> Loaded is not validated. The first adapter implementation step is to validate this contract before any target binding or simulator call.
"""
    ))


render_contract_checkpoint(historical_spec, adapter_source)

### Contract loaded for implementation work

| Item | Checked-in source |
|:---|:---|
| Historical specification | carrot-seed-opportunity-coarse-v1 |
| Schema version | 1 |
| Lifecycle | historical_reproduction |
| Parameter envelope | seed_opportunity_value: baseline 20; candidates 10.0, 40.0, 60.0, 80.0 |
| Adapter source | evaluator/runner.py |

> Loaded is not validated. The first adapter implementation step is to validate this contract before any target binding or simulator call.


## 9.2 Prototype the first adapter functions

We will design the local adapter one small function at a time in this notebook, then move a reviewed version into `evaluator/runner.py`. The first two functions have distinct jobs: `read_spec` will read one JSON file, while `validate_spec_schema` will decide whether an already-read object satisfies the versioned schema. Keeping them separate makes an I/O failure distinguishable from a contract failure.

The first small implementation is `read_spec`. It will read JSON and report only file or JSON-decoding failures; it will not decide whether the resulting value is an experiment contract. `validate_spec_schema` remains a stub for the next turn. This pass does not load a validator, choose a diagnostic/result format, bind Kaggriculture, or run a simulator.

In [183]:
from typing import Any


class SpecReadError(RuntimeError):
    """The requested specification could not be read as JSON."""


def read_spec(spec_path: Path) -> Any:
    """Read one source-controlled experiment JSON document.

    Reading is intentionally narrower than validation: it distinguishes a
    missing/unreadable file from malformed JSON, but it does not decide
    whether a valid JSON value is an approved experiment contract.
    """
    try:
        raw_specification = spec_path.read_text(encoding="utf-8")
    except OSError as error:
        raise SpecReadError(f"Could not read experiment specification: {spec_path}") from error

    try:
        return json.loads(raw_specification)
    except json.JSONDecodeError as error:
        raise SpecReadError(f"Experiment specification is not valid JSON: {spec_path}") from error


# This is the value the next step will validate; reading it does not validate it.
spec_for_schema_validation = read_spec(spec_path)
assert spec_for_schema_validation == historical_spec

In [184]:
from IPython.display import Markdown, display


def render_read_spec_summary(specification: Any, source: Path) -> None:
    """Presentation only: show facts returned by read_spec without validating them."""
    if isinstance(specification, dict):
        value_kind = "JSON object"
        top_level_keys = ", ".join(f"`{key}`" for key in specification)
        declared_identifier = specification.get("experiment_id", "not present")
    else:
        value_kind = type(specification).__name__
        top_level_keys = "not applicable"
        declared_identifier = "not applicable"

    display(Markdown(
        f"""### Specification read

| Read fact | Value |
|:---|:---|
| Source file | {source.name} |
| Decoded JSON value | {value_kind} |
| Top-level keys | {top_level_keys} |
| Declared experiment ID | {declared_identifier} |

> **Read, not validated.** These are facts returned by `read_spec`; the next function must still check them against schema v1.
"""
    ))


render_read_spec_summary(spec_for_schema_validation, spec_path)

### Specification read

| Read fact | Value |
|:---|:---|
| Source file | seed_opportunity_coarse_v1.json |
| Decoded JSON value | JSON object |
| Top-level keys | `schema_version`, `experiment_id`, `lifecycle_status`, `decision_graph`, `target`, `parameter`, `suite`, `evaluation`, `purpose`, `governance`, `historical_expectations`, `provenance` |
| Declared experiment ID | carrot-seed-opportunity-coarse-v1 |

> **Read, not validated.** These are facts returned by `read_spec`; the next function must still check them against schema v1.


In [185]:
from dataclasses import dataclass

from jsonschema import Draft202012Validator
from jsonschema.exceptions import SchemaError


class SchemaValidationSetupError(RuntimeError):
    """The checked-in schema cannot safely be used as a validator."""


@dataclass(frozen=True)
class SchemaValidationIssue:
    path: str
    rule: str
    message: str


@dataclass(frozen=True)
class SchemaValidationResult:
    schema_path: Path
    schema_draft: str
    issues: tuple[SchemaValidationIssue, ...]

    @property
    def is_valid(self) -> bool:
        return not self.issues


def validate_spec_schema(specification: Any, schema_path: Path) -> SchemaValidationResult:
    """Check an already-read specification against the checked-in schema.

    A broken schema is a project/setup failure, so it raises separately. A
    usable schema returns every contract issue as structured data instead of
    printing or deciding what the experiment should do next.
    """
    try:
        schema = json.loads(schema_path.read_text(encoding="utf-8"))
        Draft202012Validator.check_schema(schema)
    except (OSError, json.JSONDecodeError, SchemaError) as error:
        # Do not treat a damaged validator as a valid experiment with errors.
        raise SchemaValidationSetupError(f"Could not use schema: {schema_path}") from error

    validator = Draft202012Validator(schema)
    issues = tuple(
        SchemaValidationIssue(
            path=error.json_path,
            rule=str(error.validator),
            message=error.message,
        )
        for error in sorted(
            validator.iter_errors(specification),
            key=lambda error: (error.json_path, str(error.validator)),
        )
    )
    schema_draft = schema.get("$schema", "not declared") if isinstance(schema, dict) else "not declared"
    return SchemaValidationResult(schema_path, schema_draft, issues)


schema_validation = validate_spec_schema(spec_for_schema_validation, schema_path)

In [186]:
def render_schema_validation_summary(result: SchemaValidationResult) -> None:
    """Presentation only: render validation facts already held in result."""
    status = "Passed" if result.is_valid else "Failed"
    if result.is_valid:
        issue_details = "No schema-rule violations returned."
    else:
        issue_rows = "\n".join(
            f"| `{issue.path}` | `{issue.rule}` | {issue.message} |"
            for issue in result.issues
        )
        issue_details = f"""\n\n| JSON path | Rule | Message |
|:---|:---|:---|
{issue_rows}"""

    display(Markdown(
        f"""### Schema validation

| Check | Result |
|:---|:---|
| Schema file | {result.schema_path.name} |
| Draft | {result.schema_draft} |
| Contract status | **{status}** |
| Reported issues | {len(result.issues)} |

{issue_details}

> This checks contract shape only. Lifecycle authorization, target fingerprinting, and simulator execution remain separate steps.
"""
    ))


render_schema_validation_summary(schema_validation)

### Schema validation

| Check | Result |
|:---|:---|
| Schema file | experiment-spec-v1.schema.json |
| Draft | https://json-schema.org/draft/2020-12/schema |
| Contract status | **Passed** |
| Reported issues | 0 |

No schema-rule violations returned.

> This checks contract shape only. Lifecycle authorization, target fingerprinting, and simulator execution remain separate steps.


### Example: a malformed contract

This in-memory example removes the required `suite` field. The checked-in historical JSON is not changed.

In [187]:
# Copy only the top-level mapping so this example cannot mutate the loaded contract.
malformed_specification = dict(spec_for_schema_validation)
del malformed_specification["suite"]

malformed_schema_validation = validate_spec_schema(
    malformed_specification, schema_path
)
assert not malformed_schema_validation.is_valid

In [188]:
render_schema_validation_summary(malformed_schema_validation)

### Schema validation

| Check | Result |
|:---|:---|
| Schema file | experiment-spec-v1.schema.json |
| Draft | https://json-schema.org/draft/2020-12/schema |
| Contract status | **Failed** |
| Reported issues | 1 |



| JSON path | Rule | Message |
|:---|:---|:---|
| `$` | `required` | 'suite' is a required property |

> This checks contract shape only. Lifecycle authorization, target fingerprinting, and simulator execution remain separate steps.


### Next prototype: explicit target fingerprint

This stage binds only the target path supplied to the run request, then compares its Git revision, carrot-policy hash, and engine hash to the already schema-valid contract. Revision alone cannot detect uncommitted edits, so all three facts must match. A mismatch identifies a different system under test and stops later stages before simulation. It does not authorize a lifecycle or run a simulator.

In [189]:
from dataclasses import dataclass


# The proven parent binding is deliberately imported from the parent source tree.
# It receives an explicit path below; it is never asked to guess a target from cwd.
parent_evaluator_source = project_root.parent / "src"
if str(parent_evaluator_source) not in sys.path:
    sys.path.insert(0, str(parent_evaluator_source))

from kaggriculture_eval.binding import bind_target


@dataclass(frozen=True)
class TargetFingerprintCheck:
    label: str
    expected: str
    observed: str

    @property
    def matches(self) -> bool:
        return self.expected == self.observed


@dataclass(frozen=True)
class TargetFingerprintValidation:
    target_root: Path
    binding: Any | None
    binding_error: str | None
    checks: tuple[TargetFingerprintCheck, ...]

    @property
    def is_match(self) -> bool:
        return self.binding_error is None and all(check.matches for check in self.checks)


def bind_and_validate_target(
    specification: dict[str, Any], target_root: Path
) -> TargetFingerprintValidation:
    """Bind the explicit target and compare it with the contract fingerprint.

    A missing/broken target becomes structured evidence for the presentation
    cell. A successful binding retains the parent evaluator's object for a
    later runner stage, but this function does not run the simulator.
    """
    try:
        # Passing target_root, rather than None, prevents environment variables
        # or the current directory from silently selecting another checkout.
        binding = bind_target(target_root)
    except RuntimeError as error:
        return TargetFingerprintValidation(
            target_root=target_root.resolve(),
            binding=None,
            binding_error=str(error),
            checks=(),
        )

    expected = specification["target"]
    # Git revision identifies committed history; content hashes also catch
    # relevant uncommitted policy or engine edits at that same revision.
    checks = (
        TargetFingerprintCheck(
            "Git revision", expected["expected_revision"], binding.revision
        ),
        TargetFingerprintCheck(
            "Carrot policy SHA-256",
            expected["expected_shared_sha256"],
            binding.shared_sha256,
        ),
        TargetFingerprintCheck(
            "Game engine SHA-256",
            expected["expected_engine_sha256"],
            binding.engine_sha256,
        ),
    )
    return TargetFingerprintValidation(
        target_root=binding.root,
        binding=binding,
        binding_error=None,
        checks=checks,
    )


# Kaggriculture is a sibling of foundry-trainer, not of carrot-tuning-system.
explicit_target_root = project_root.parent.parent / "kaggriculture"
target_fingerprint_validation = bind_and_validate_target(
    spec_for_schema_validation, explicit_target_root
)
assert target_fingerprint_validation.is_match

In [190]:
from IPython.display import Markdown, display


def render_target_fingerprint_summary(result: TargetFingerprintValidation) -> None:
    """Presentation only: render target-binding facts already in result."""
    if result.binding_error is not None:
        detail = f"**Binding failed:** {result.binding_error}"
    else:
        rows = "\n".join(
            f"| {check.label} | `{check.expected}` | `{check.observed}` | {'yes' if check.matches else 'no'} |"
            for check in result.checks
        )
        detail = f"""| Target fact | Expected | Observed | Match |
|:---|:---|:---|:---|
{rows}"""

    status = "Exact target match" if result.is_match else "Target mismatch"
    display(Markdown(
        f"""### Target fingerprint

| Check | Result |
|:---|:---|
| Explicit target directory | {result.target_root} |
| Binding and fingerprint status | **{status}** |

{detail}

> This proves which checkout would be evaluated; it does not authorize a lifecycle or start a simulator.
"""
    ))


render_target_fingerprint_summary(target_fingerprint_validation)

### Target fingerprint

| Check | Result |
|:---|:---|
| Explicit target directory | /home/experiments/frontier-week/kaggriculture |
| Binding and fingerprint status | **Exact target match** |

| Target fact | Expected | Observed | Match |
|:---|:---|:---|:---|
| Git revision | `6ace5fed9018fef45642b41ea11d0bd741a508a4` | `6ace5fed9018fef45642b41ea11d0bd741a508a4` | yes |
| Carrot policy SHA-256 | `9abb01678ad20a30927594a92c5234d8db798c85a2e8bc85391fbb72b4bf5a36` | `9abb01678ad20a30927594a92c5234d8db798c85a2e8bc85391fbb72b4bf5a36` | yes |
| Game engine SHA-256 | `bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d01f463f23bccee653e` | `bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d01f463f23bccee653e` | yes |

> This proves which checkout would be evaluated; it does not authorize a lifecycle or start a simulator.


### Next prototype: lifecycle authorization

The schema can tell us that a lifecycle value is one of the allowed words, but this stage decides whether that word permits a local simulator run. An `approved` contract may proceed to later local checks. A `historical_reproduction` record is evidence and a regression oracle, not automatic permission to rerun. Draft and retired contracts also stop here. This stage does not modify policy source.

In [191]:
@dataclass(frozen=True)
class LifecycleAuthorization:
    declared_status: str
    is_authorized: bool
    reason: str
    human_decision_required: bool
    source_policy_mutation_authorized: bool
    candidate_expansion_authorized: bool


def authorize_lifecycle(specification: dict[str, Any]) -> LifecycleAuthorization:
    """Interpret a schema-valid lifecycle without running the experiment.

    The status is a narrow execution gate, not a source-edit permission.
    Governance flags are carried into the result so later stages cannot
    mistake an allowed local run for permission to mutate policy or widen
    the approved candidate envelope.
    """
    status = specification["lifecycle_status"]
    governance = specification["governance"]
    reasons = {
        "approved": "The contract is explicitly approved for bounded local execution.",
        "historical_reproduction": (
            "Historical evidence is a regression oracle, not automatic rerun permission."
        ),
        "draft": "A draft contract has not received human approval.",
        "retired": "A retired contract must not be run again.",
    }
    return LifecycleAuthorization(
        declared_status=status,
        is_authorized=status == "approved",
        reason=reasons[status],
        human_decision_required=governance["human_decision_required"],
        source_policy_mutation_authorized=governance["source_policy_mutation_authorized"],
        candidate_expansion_authorized=governance["candidate_expansion_authorized"],
    )


lifecycle_authorization = authorize_lifecycle(spec_for_schema_validation)
assert not lifecycle_authorization.is_authorized

In [192]:
from IPython.display import Markdown, display


def render_lifecycle_authorization_summary(result: LifecycleAuthorization) -> None:
    """Presentation only: render lifecycle facts already held in result."""
    authorized = "Yes" if result.is_authorized else "No"
    human_decision = "yes" if result.human_decision_required else "no"
    source_mutation = "yes" if result.source_policy_mutation_authorized else "no"
    expansion = "yes" if result.candidate_expansion_authorized else "no"
    display(Markdown(
        f"""### Lifecycle authorization

| Check | Result |
|:---|:---|
| Declared lifecycle | `{result.declared_status}` |
| Authorized for a local run | **{authorized}** |
| Human decision required | {human_decision} |
| Source-policy mutation authorized | {source_mutation} |
| Candidate expansion authorized | {expansion} |

> {result.reason} The adapter stops before simulation when authorization is absent.
"""
    ))


render_lifecycle_authorization_summary(lifecycle_authorization)

### Lifecycle authorization

| Check | Result |
|:---|:---|
| Declared lifecycle | `historical_reproduction` |
| Authorized for a local run | **No** |
| Human decision required | yes |
| Source-policy mutation authorized | no |
| Candidate expansion authorized | no |

> Historical evidence is a regression oracle, not automatic rerun permission. The adapter stops before simulation when authorization is absent.


### Next prototype: local-control validation

This stage checks that the already schema-valid declaration still names a real, permitted one-field carrot-policy override and a complete deterministic local suite. It compares the declared baseline with the actual target-policy default, requires a predeclared non-baseline candidate list, and preserves checkpoint, audit, and governance locks. It is a contract check only: it does not override the historical lifecycle stop or run a game.

In [193]:
from dataclasses import fields


@dataclass(frozen=True)
class LocalControlCheck:
    label: str
    passed: bool
    detail: str


@dataclass(frozen=True)
class LocalControlValidation:
    parameter_name: str
    declared_baseline: float
    target_default: float | None
    candidate_count: int
    checks: tuple[LocalControlCheck, ...]

    @property
    def is_valid(self) -> bool:
        return all(check.passed for check in self.checks)


def validate_local_controls(
    specification: dict[str, Any], binding: Any
) -> LocalControlValidation:
    """Check the bounded in-memory control envelope against the bound target.

    The target binding is already fingerprinted, so the policy class below
    is attributable to the approved checkout. This function inspects it; it
    never constructs an override, changes source, or calls a simulator.
    """
    parameter = specification["parameter"]
    suite = specification["suite"]
    evaluation = specification["evaluation"]
    governance = specification["governance"]
    parameter_class = binding.shared.CarrotParameters
    parameter_fields = {field.name: field for field in fields(parameter_class)}

    parameter_name = parameter["name"]
    candidate_values = parameter["candidate_values"]
    parameter_exists = parameter_name in parameter_fields
    target_default = (
        parameter_fields[parameter_name].default if parameter_exists else None
    )
    known_conveyor = hasattr(binding.shared, "carrot_conveyor_agent")

    checks = (
        LocalControlCheck(
            "Parameter namespace",
            specification["decision_graph"]["parameter_namespace"] == parameter_class.__name__,
            f"declared {specification['decision_graph']['parameter_namespace']}; target {parameter_class.__name__}",
        ),
        LocalControlCheck(
            "Declared parameter exists",
            parameter_exists,
            parameter_name if parameter_exists else f"unknown field: {parameter_name}",
        ),
        LocalControlCheck(
            "Baseline matches target default",
            parameter_exists and parameter["baseline_value"] == target_default,
            f"declared {parameter['baseline_value']}; target {target_default}",
        ),
        LocalControlCheck(
            "Candidate envelope",
            bool(candidate_values)
            and len(candidate_values) == len(set(candidate_values))
            and all(value != parameter["baseline_value"] for value in candidate_values),
            f"{len(candidate_values)} predeclared values; none may equal the baseline",
        ),
        LocalControlCheck(
            "Mutation mode",
            parameter["mutation_mode"] == "in_memory_single_field_override",
            parameter["mutation_mode"],
        ),
        LocalControlCheck(
            "Deterministic known opponent",
            suite["opponent_deterministic"]
            and suite["opponent_id"] == "target.agents.carrot.shared.carrot_conveyor_agent"
            and known_conveyor,
            suite["opponent_id"],
        ),
        LocalControlCheck(
            "Complete matched suite",
            bool(suite["seeds"])
            and len(suite["seeds"]) == len(set(suite["seeds"]))
            and set(suite["seats"]) == {0, 1}
            and suite["steps"] > 0,
            f"{len(suite['seeds'])} seeds; seats {suite['seats']}; {suite['steps']} steps",
        ),
        LocalControlCheck(
            "Checkpoint and audit policy",
            evaluation["checkpoint_policy"] == "before_every_decision"
            and evaluation["reuse_no_divergence_baseline"]
            and evaluation["clean_audit_required"]
            and evaluation["result_storage"] == "local_ignored_var_directory",
            "checkpoint replay with required clean audits and local ignored output",
        ),
        LocalControlCheck(
            "Governance locks",
            governance["human_decision_required"]
            and not governance["source_policy_mutation_authorized"]
            and not governance["candidate_expansion_authorized"],
            "human decision required; no source mutation or candidate expansion",
        ),
    )
    return LocalControlValidation(
        parameter_name=parameter_name,
        declared_baseline=parameter["baseline_value"],
        target_default=target_default,
        candidate_count=len(candidate_values),
        checks=checks,
    )


assert target_fingerprint_validation.binding is not None
local_control_validation = validate_local_controls(
    spec_for_schema_validation, target_fingerprint_validation.binding
)
assert local_control_validation.is_valid

In [194]:
from IPython.display import Markdown, display


def render_local_control_summary(result: LocalControlValidation) -> None:
    """Presentation only: render local-control facts already in result."""
    status = "Passed" if result.is_valid else "Failed"
    rows = "\n".join(
        f"| {check.label} | {'pass' if check.passed else 'fail'} | {check.detail} |"
        for check in result.checks
    )
    display(Markdown(
        f"""### Local control validation

| Parameter fact | Result |
|:---|:---|
| Declared field | `{result.parameter_name}` |
| Declared baseline / target default | {result.declared_baseline} / {result.target_default} |
| Predeclared candidates | {result.candidate_count} |
| Overall status | **{status}** |

| Control | Result | Detail |
|:---|:---|:---|
{rows}

> These controls describe the permitted local envelope; lifecycle authorization and simulator execution remain separate gates.
"""
    ))


render_local_control_summary(local_control_validation)

### Local control validation

| Parameter fact | Result |
|:---|:---|
| Declared field | `seed_opportunity_value` |
| Declared baseline / target default | 20.0 / 20.0 |
| Predeclared candidates | 4 |
| Overall status | **Passed** |

| Control | Result | Detail |
|:---|:---|:---|
| Parameter namespace | pass | declared CarrotParameters; target CarrotParameters |
| Declared parameter exists | pass | seed_opportunity_value |
| Baseline matches target default | pass | declared 20.0; target 20.0 |
| Candidate envelope | pass | 4 predeclared values; none may equal the baseline |
| Mutation mode | pass | in_memory_single_field_override |
| Deterministic known opponent | pass | target.agents.carrot.shared.carrot_conveyor_agent |
| Complete matched suite | pass | 30 seeds; seats [0, 1]; 720 steps |
| Checkpoint and audit policy | pass | checkpoint replay with required clean audits and local ignored output |
| Governance locks | pass | human decision required; no source mutation or candidate expansion |

> These controls describe the permitted local envelope; lifecycle authorization and simulator execution remain separate gates.


### Next prototype: deterministic baseline-cache plan

Before a baseline can be reused, the adapter needs a stable way to identify exactly what it represents. This prototype derives one content-addressed cache group from the experiment ID, bound-target fingerprint, baseline parameter value, suite, and checkpoint policy; within it, it plans one cache path for every seed/seat match. It does **not** create directories or files, call `baseline_match`, or run Kaggriculture. A later reviewed step may materialize only this plan's missing entries.

In [195]:
import hashlib


@dataclass(frozen=True)
class BaselineCacheEntry:
    """One planned baseline cache; its path need not exist yet."""

    seed: int
    seat: int
    path: Path


@dataclass(frozen=True)
class BaselineCachePlan:
    """A deterministic description of baseline caches, not cache contents."""

    group_id: str
    cache_directory: Path
    entries: tuple[BaselineCacheEntry, ...]


def plan_baseline_caches(
    specification: dict[str, Any], binding: Any, cache_root: Path
) -> BaselineCachePlan:
    """Derive cache identities and paths without writing or evaluating anything."""
    suite = specification["suite"]
    parameter = specification["parameter"]

    # Every fact that could make a baseline unsafe to reuse belongs in its identity.
    identity = {
        "cache_format": "baseline-match-v1",
        "experiment_id": specification["experiment_id"],
        "target_revision": binding.revision,
        "shared_sha256": binding.shared_sha256,
        "engine_sha256": binding.engine_sha256,
        "baseline_parameter": {
            parameter["name"]: parameter["baseline_value"]
        },
        "suite": {
            "opponent_id": suite["opponent_id"],
            "seeds": suite["seeds"],
            "seats": suite["seats"],
            "steps": suite["steps"],
        },
        "checkpoint_policy": specification["evaluation"]["checkpoint_policy"],
    }
    canonical_identity = json.dumps(identity, sort_keys=True, separators=(",", ":"))
    group_id = hashlib.sha256(canonical_identity.encode("utf-8")).hexdigest()
    cache_directory = cache_root / "baseline-caches" / group_id

    entries = tuple(
        BaselineCacheEntry(
            seed=seed,
            seat=seat,
            path=cache_directory / f"seed-{seed}-seat-{seat}.json.gz",
        )
        for seed in suite["seeds"]
        for seat in suite["seats"]
    )
    return BaselineCachePlan(group_id, cache_directory, entries)


assert target_fingerprint_validation.binding is not None
baseline_cache_plan = plan_baseline_caches(
    spec_for_schema_validation,
    target_fingerprint_validation.binding,
    project_root.parent / "var" / "carrot-tuning-system",
)
assert len(baseline_cache_plan.entries) == 60


In [196]:
def render_baseline_cache_plan(plan: BaselineCachePlan) -> None:
    """Presentation only: render already-derived cache-plan facts."""
    first_entry = plan.entries[0]
    last_entry = plan.entries[-1]
    display(Markdown(
        f"""### Planned baseline cache set

| Fact | Value |
|:---|:---|
| Cache-group ID | `{plan.group_id[:12]}…` |
| Planned baseline matches | {len(plan.entries)} ({len(plan.entries) // 2} seeds × 2 seats) |
| First planned cache | `{first_entry.path.name}` |
| Last planned cache | `{last_entry.path.name}` |
| Files created by this cell | No |

> This is a deterministic plan only. It has not created a directory or cache file, and it has not run a baseline game.
"""
    ))


render_baseline_cache_plan(baseline_cache_plan)


### Planned baseline cache set

| Fact | Value |
|:---|:---|
| Cache-group ID | `652491086c28…` |
| Planned baseline matches | 60 (30 seeds × 2 seats) |
| First planned cache | `seed-42-seat-0.json.gz` |
| Last planned cache | `seed-71-seat-1.json.gz` |
| Files created by this cell | No |

> This is a deterministic plan only. It has not created a directory or cache file, and it has not run a baseline game.


### Next prototype: baseline-cache state preflight

A deterministic path alone does not mean a baseline is reusable. This read-only preflight partitions the planned paths into files present and files missing, so a later materializer can be limited to the missing entries. A present file is only a candidate for reuse: a later step must read and validate its content before trusting it. This prototype does not create a directory, read a cache payload, call `baseline_match`, or run Kaggriculture.

In [197]:
@dataclass(frozen=True)
class BaselineCacheState:
    """Read-only presence facts for one planned baseline-cache group."""

    plan: BaselineCachePlan
    present_entries: tuple[BaselineCacheEntry, ...]
    missing_entries: tuple[BaselineCacheEntry, ...]

    @property
    def is_complete(self) -> bool:
        return not self.missing_entries


def inspect_baseline_cache_state(plan: BaselineCachePlan) -> BaselineCacheState:
    """Classify planned paths by file presence without opening or creating them."""
    # Presence is deliberately weaker than validity; payload validation comes later.
    present_entries = tuple(entry for entry in plan.entries if entry.path.is_file())
    missing_entries = tuple(entry for entry in plan.entries if not entry.path.is_file())
    return BaselineCacheState(plan, present_entries, missing_entries)


baseline_cache_state = inspect_baseline_cache_state(baseline_cache_plan)
assert len(baseline_cache_state.present_entries) + len(baseline_cache_state.missing_entries) == 60


In [198]:
def render_baseline_cache_state(state: BaselineCacheState) -> None:
    """Presentation only: render read-only cache-presence facts."""
    display(Markdown(
        f"""### Baseline-cache preflight

| Fact | Value |
|:---|:---|
| Planned baseline files | {len(state.plan.entries)} |
| Files present | {len(state.present_entries)} |
| Files missing | {len(state.missing_entries)} |
| Complete cache group | {'Yes' if state.is_complete else 'No'} |

> Presence is not validity: no cache payload was opened or trusted. This cell made no files and ran no baseline game.
"""
    ))


render_baseline_cache_state(baseline_cache_state)


### Baseline-cache preflight

| Fact | Value |
|:---|:---|
| Planned baseline files | 60 |
| Files present | 0 |
| Files missing | 60 |
| Complete cache group | No |

> Presence is not validity: no cache payload was opened or trusted. This cell made no files and ran no baseline game.


### Next prototype: structural cache-payload validation

Before a present file can be reused, its decoded cache payload must still describe this exact target and planned match. This prototype checks the baseline-cache schema marker, kind, bound-target fingerprint, seed/seat/step/opponent identity, checkpoint-record shape, and outcome shape. It accepts an already-decoded mapping; it does not open gzip files, restore a checkpoint, create a cache, or run Kaggriculture. The example below is an in-memory structural example, not simulator evidence.

In [199]:
from copy import deepcopy


@dataclass(frozen=True)
class BaselineCachePayloadCheck:
    label: str
    passed: bool
    detail: str


@dataclass(frozen=True)
class BaselineCachePayloadValidation:
    entry: BaselineCacheEntry
    checks: tuple[BaselineCachePayloadCheck, ...]

    @property
    def is_valid(self) -> bool:
        return all(check.passed for check in self.checks)


def validate_baseline_cache_payload(
    cache: dict[str, Any],
    entry: BaselineCacheEntry,
    specification: dict[str, Any],
    binding: Any,
) -> BaselineCachePayloadValidation:
    """Check decoded cache provenance and shape without restoring a game state."""
    cache_binding = cache.get("binding")
    match = cache.get("match")
    turns = cache.get("turns")
    outcome = cache.get("outcome")
    expected_outcome_keys = {
        "decision_reward", "opponent_reward",
        "decision_status", "opponent_status",
    }

    checks = (
        BaselineCachePayloadCheck(
            "Cache schema and kind",
            cache.get("schema_version") == 1 and cache.get("kind") == "baseline-match",
            f"schema {cache.get('schema_version')}; kind {cache.get('kind')}",
        ),
        BaselineCachePayloadCheck(
            "Bound target fingerprint",
            isinstance(cache_binding, dict)
            and cache_binding.get("target_root") == str(binding.root)
            and cache_binding.get("target_revision") == binding.revision
            and cache_binding.get("shared_sha256") == binding.shared_sha256
            and cache_binding.get("engine_sha256") == binding.engine_sha256,
            "target root, revision, and both source hashes must match",
        ),
        BaselineCachePayloadCheck(
            "Planned match identity",
            isinstance(match, dict)
            and match.get("seed") == entry.seed
            and match.get("decision_player") == entry.seat
            and match.get("steps") == specification["suite"]["steps"]
            and match.get("opponent") == specification["suite"]["opponent_id"]
            and match.get("policy") == "target.agents.carrot.shared.carrot_decision_agent"
            and match.get("configuration") == {
                "episodeSteps": specification["suite"]["steps"],
                "seed": entry.seed,
            },
            f"seed {entry.seed}; seat {entry.seat}; {specification['suite']['steps']} steps",
        ),
        BaselineCachePayloadCheck(
            "Checkpoint-record shape",
            isinstance(turns, list)
            and bool(turns)
            and all(
                isinstance(turn, dict)
                and {"turn", "checkpoint_state", "observation", "baseline_action"} <= set(turn)
                for turn in turns
            ),
            "nonempty records with checkpoint, observation, and baseline action",
        ),
        BaselineCachePayloadCheck(
            "Outcome shape",
            isinstance(outcome, dict) and expected_outcome_keys <= set(outcome),
            "decision/opponent reward and status fields",
        ),
    )
    return BaselineCachePayloadValidation(entry, checks)


assert target_fingerprint_validation.binding is not None
example_entry = baseline_cache_plan.entries[0]
illustrative_cache_payload = {
    "schema_version": 1,
    "kind": "baseline-match",
    "binding": {
        "target_root": str(target_fingerprint_validation.binding.root),
        "target_revision": target_fingerprint_validation.binding.revision,
        "shared_sha256": target_fingerprint_validation.binding.shared_sha256,
        "engine_sha256": target_fingerprint_validation.binding.engine_sha256,
    },
    "match": {
        "seed": example_entry.seed,
        "decision_player": example_entry.seat,
        "steps": spec_for_schema_validation["suite"]["steps"],
        "opponent": spec_for_schema_validation["suite"]["opponent_id"],
        "policy": "target.agents.carrot.shared.carrot_decision_agent",
        "configuration": {"episodeSteps": 720, "seed": example_entry.seed},
    },
    "turns": [{
        "turn": 0, "checkpoint_state": [], "observation": {}, "baseline_action": {},
    }],
    "outcome": {
        "decision_reward": 0, "opponent_reward": 0,
        "decision_status": "DONE", "opponent_status": "DONE",
    },
}
cache_payload_validation = validate_baseline_cache_payload(
    illustrative_cache_payload,
    example_entry,
    spec_for_schema_validation,
    target_fingerprint_validation.binding,
)
assert cache_payload_validation.is_valid

wrong_seed_payload = deepcopy(illustrative_cache_payload)
wrong_seed_payload["match"]["seed"] += 1
wrong_seed_validation = validate_baseline_cache_payload(
    wrong_seed_payload, example_entry, spec_for_schema_validation, target_fingerprint_validation.binding
)
assert not wrong_seed_validation.is_valid


In [200]:
def render_cache_payload_validation(result: BaselineCachePayloadValidation) -> None:
    """Presentation only: render existing structural-validation facts."""
    rows = "\n".join(
        f"| {check.label} | {'pass' if check.passed else 'fail'} |"
        for check in result.checks
    )
    display(Markdown(
        f"""### Illustrative cache-payload validation

| Check | Result |
|:---|:---|
{rows}

> This pass applies only to an in-memory illustrative structure. It is not a decoded baseline file or simulator evidence. The altered-seed example fails the planned-match check.
"""
    ))


render_cache_payload_validation(cache_payload_validation)


### Illustrative cache-payload validation

| Check | Result |
|:---|:---|
| Cache schema and kind | pass |
| Bound target fingerprint | pass |
| Planned match identity | pass |
| Checkpoint-record shape | pass |
| Outcome shape | pass |

> This pass applies only to an in-memory illustrative structure. It is not a decoded baseline file or simulator evidence. The altered-seed example fails the planned-match check.


### Next prototype: read and validate one present gzip cache

The structural validator needs a decoded payload. This component combines the proven parent `read_baseline` gzip reader with the reviewed structural validator, returning a structured result for missing files, unreadable files, and valid or invalid decoded payloads. It does not create or materialize a baseline. Its exercise below writes a **temporary synthetic fixture** solely to prove the reader/validator handoff; that file is removed before the cell finishes and is not simulator evidence.

In [201]:
from tempfile import TemporaryDirectory

from kaggriculture_eval.checkpoint import read_baseline, write_baseline


@dataclass(frozen=True)
class BaselineCacheReadResult:
    """Read and structural-validation facts for one planned gzip cache."""

    entry: BaselineCacheEntry
    file_was_present: bool
    read_error: str | None
    payload_validation: BaselineCachePayloadValidation | None

    @property
    def is_reusable_candidate(self) -> bool:
        return (
            self.file_was_present
            and self.read_error is None
            and self.payload_validation is not None
            and self.payload_validation.is_valid
        )


def read_and_validate_baseline_cache(
    entry: BaselineCacheEntry, specification: dict[str, Any], binding: Any
) -> BaselineCacheReadResult:
    """Read one present gzip cache and validate it without restoring game state."""
    if not entry.path.is_file():
        return BaselineCacheReadResult(entry, False, None, None)

    try:
        cache = read_baseline(entry.path)
    except (OSError, ValueError, json.JSONDecodeError) as error:
        return BaselineCacheReadResult(entry, True, str(error), None)

    payload_validation = validate_baseline_cache_payload(
        cache, entry, specification, binding
    )
    return BaselineCacheReadResult(entry, True, None, payload_validation)


with TemporaryDirectory() as temporary_directory:
    # The fixture exercises gzip decoding only; it was not produced by a simulator.
    synthetic_entry = BaselineCacheEntry(
        seed=example_entry.seed,
        seat=example_entry.seat,
        path=Path(temporary_directory) / "synthetic-baseline.json.gz",
    )
    write_baseline(illustrative_cache_payload, synthetic_entry.path)
    gzip_cache_read_result = read_and_validate_baseline_cache(
        synthetic_entry,
        spec_for_schema_validation,
        target_fingerprint_validation.binding,
    )

    wrong_seed_fixture = deepcopy(illustrative_cache_payload)
    wrong_seed_fixture["match"]["seed"] += 1
    wrong_seed_entry = BaselineCacheEntry(
        seed=example_entry.seed,
        seat=example_entry.seat,
        path=Path(temporary_directory) / "wrong-seed-baseline.json.gz",
    )
    write_baseline(wrong_seed_fixture, wrong_seed_entry.path)
    wrong_seed_gzip_result = read_and_validate_baseline_cache(
        wrong_seed_entry,
        spec_for_schema_validation,
        target_fingerprint_validation.binding,
    )

    assert gzip_cache_read_result.is_reusable_candidate
    assert not wrong_seed_gzip_result.is_reusable_candidate
    assert wrong_seed_gzip_result.payload_validation is not None
    assert any(
        check.label == "Planned match identity" and not check.passed
        for check in wrong_seed_gzip_result.payload_validation.checks
    )


In [202]:
def render_gzip_cache_read_result(result: BaselineCacheReadResult) -> None:
    """Presentation only: render existing gzip-reader handoff facts."""
    validation_status = (
        "Passed" if result.payload_validation and result.payload_validation.is_valid else "Not passed"
    )
    display(Markdown(
        f"""### Gzip cache reader handoff

| Fact | Result |
|:---|:---|
| File was present while read | {'Yes' if result.file_was_present else 'No'} |
| Gzip read error | {result.read_error or 'None'} |
| Structural payload validation | {validation_status} |
| Reusable candidate | {'Yes' if result.is_reusable_candidate else 'No'} |

> The successful fixture is synthetic and temporary, not a baseline game or durable cache. A second temporary fixture with the wrong seed is read successfully but rejected by structural validation.
"""
    ))


render_gzip_cache_read_result(gzip_cache_read_result)


### Gzip cache reader handoff

| Fact | Result |
|:---|:---|
| File was present while read | Yes |
| Gzip read error | None |
| Structural payload validation | Passed |
| Reusable candidate | Yes |

> The successful fixture is synthetic and temporary, not a baseline game or durable cache. A second temporary fixture with the wrong seed is read successfully but rejected by structural validation.


### Next prototype: cache reuse decision

This component turns the previous facts into an explicit, fail-closed plan. A planned entry is reusable only when it was present, decoded without error, and passed structural validation. A missing, unreadable, invalid, or skipped-present entry is marked for **later** materialization with its reason. The result is a decision record, not a materializer: it creates no cache, restores no checkpoint, and runs no game.

In [203]:
@dataclass(frozen=True)
class BaselineCacheAction:
    """One reuse-or-materialize-later decision with an auditable reason."""

    entry: BaselineCacheEntry
    decision: str
    reason: str


@dataclass(frozen=True)
class BaselineCachePreparationPlan:
    """A complete, non-executing disposition for one cache group."""

    cache_plan: BaselineCachePlan
    actions: tuple[BaselineCacheAction, ...]

    @property
    def reusable_actions(self) -> tuple[BaselineCacheAction, ...]:
        return tuple(action for action in self.actions if action.decision == "reuse")

    @property
    def materialize_actions(self) -> tuple[BaselineCacheAction, ...]:
        return tuple(action for action in self.actions if action.decision == "materialize")


def plan_baseline_cache_preparation(
    state: BaselineCacheState, read_results: tuple[BaselineCacheReadResult, ...]
) -> BaselineCachePreparationPlan:
    """Choose only reuse or later materialization; do not execute either path."""
    results_by_path = {result.entry.path: result for result in read_results}
    present_paths = {entry.path for entry in state.present_entries}
    actions = []

    for entry in state.plan.entries:
        if entry.path not in present_paths:
            actions.append(BaselineCacheAction(entry, "materialize", "cache file is missing"))
            continue

        result = results_by_path.get(entry.path)
        if result is None:
            actions.append(BaselineCacheAction(
                entry, "materialize", "present cache was not read and validated"
            ))
        elif result.is_reusable_candidate:
            actions.append(BaselineCacheAction(
                entry, "reuse", "gzip decoded and structural checks passed"
            ))
        elif result.read_error is not None:
            actions.append(BaselineCacheAction(
                entry, "materialize", f"cache could not be read: {result.read_error}"
            ))
        else:
            actions.append(BaselineCacheAction(
                entry, "materialize", "decoded cache failed structural validation"
            ))

    return BaselineCachePreparationPlan(state.plan, tuple(actions))


baseline_cache_preparation = plan_baseline_cache_preparation(
    baseline_cache_state, ()
)
assert len(baseline_cache_preparation.materialize_actions) == 60
assert not baseline_cache_preparation.reusable_actions

# This compact in-memory state demonstrates both possible dispositions; it creates nothing.
demonstration_missing_entry = baseline_cache_plan.entries[1]
demonstration_plan = BaselineCachePlan(
    "temporary-demonstration",
    Path("/temporary-demonstration"),
    (gzip_cache_read_result.entry, wrong_seed_gzip_result.entry, demonstration_missing_entry),
)
demonstration_state = BaselineCacheState(
    demonstration_plan,
    (gzip_cache_read_result.entry, wrong_seed_gzip_result.entry),
    (demonstration_missing_entry,),
)
demonstration_preparation = plan_baseline_cache_preparation(
    demonstration_state,
    (gzip_cache_read_result, wrong_seed_gzip_result),
)
assert [action.decision for action in demonstration_preparation.actions] == [
    "reuse", "materialize", "materialize",
]


In [204]:
def render_baseline_cache_preparation(plan: BaselineCachePreparationPlan) -> None:
    """Presentation only: render existing reuse/materialize decisions."""
    display(Markdown(
        f"""### Baseline-cache preparation

| Decision | Count |
|:---|---:|
| Reuse existing validated cache | {len(plan.reusable_actions)} |
| Materialize later | {len(plan.materialize_actions)} |

> The current cache group has no present files, so every entry remains planned for later materialization. This cell only records the decision; it does not create a baseline or run a game.
"""
    ))


render_baseline_cache_preparation(baseline_cache_preparation)


### Baseline-cache preparation

| Decision | Count |
|:---|---:|
| Reuse existing validated cache | 0 |
| Materialize later | 60 |

> The current cache group has no present files, so every entry remains planned for later materialization. This cell only records the decision; it does not create a baseline or run a game.


### Next prototype: materialize one selected missing baseline

This is the first component that would run Kaggriculture and write a durable local cache, so the notebook defines it but does not call it. It accepts one action already marked `materialize`, refuses to overwrite an existing file, runs the proven baseline evaluator for that action's seed/seat/step contract, and writes the resulting gzip cache. The current historical record is still `historical_reproduction`, not an approved execution request, so this definition is a review artifact only. A future call needs both an approved experiment specification and an explicit human decision to execute the selected action.

In [205]:
@dataclass(frozen=True)
class BaselineMaterializationResult:
    """Facts returned only after one baseline has actually been written."""

    action: BaselineCacheAction
    cache_path: Path
    checkpoint_count: int
    outcome: dict[str, Any]


def materialize_selected_baseline(
    action: BaselineCacheAction, specification: dict[str, Any], binding: Any
) -> BaselineMaterializationResult:
    """Run and persist exactly one approved missing baseline cache.

    This function has deliberate side effects and is intentionally not called
    in this notebook yet. The reusable runner may call it only after its
    lifecycle and preparation gates have selected this action.
    """
    if action.decision != "materialize":
        raise ValueError("only a materialize action may create a baseline cache")
    if action.entry.path.exists():
        raise FileExistsError(f"refusing to overwrite existing cache: {action.entry.path}")

    # Importing here keeps the notebook's definition and preview simulator-free.
    from kaggriculture_eval.checkpoint import baseline_match, write_baseline

    suite = specification["suite"]
    cache = baseline_match(
        binding,
        seed=action.entry.seed,
        seat=action.entry.seat,
        steps=suite["steps"],
        opponent=binding.shared.carrot_conveyor_agent,
        opponent_id=suite["opponent_id"],
    )
    write_baseline(cache, action.entry.path)
    return BaselineMaterializationResult(
        action=action,
        cache_path=action.entry.path,
        checkpoint_count=len(cache["turns"]),
        outcome=cache["outcome"],
    )


selected_materialization_action = baseline_cache_preparation.materialize_actions[0]
assert selected_materialization_action.entry.seed == 42
assert selected_materialization_action.entry.seat == 0
assert not selected_materialization_action.entry.path.exists()
# Do not call materialize_selected_baseline here.


In [206]:
def render_materialization_preview(
    action: BaselineCacheAction, lifecycle: LifecycleAuthorization
) -> None:
    """Presentation only: render the selected action and current authority."""
    display(Markdown(
        f"""### Selected baseline materialization — preview only

| Fact | Value |
|:---|:---|
| Selected match | seed {action.entry.seed}, seat {action.entry.seat}, {spec_for_schema_validation['suite']['steps']} steps |
| Decision reason | {action.reason} |
| Cache filename | `{action.entry.path.name}` |
| Current lifecycle permits execution | {'Yes' if lifecycle.is_authorized else 'No'} (`{lifecycle.declared_status}`) |
| Baseline game run by this cell | No |

> This is a preview of the one selected action. Execution remains blocked until a separately approved experiment specification and explicit human decision authorize it.
"""
    ))


render_materialization_preview(selected_materialization_action, lifecycle_authorization)


### Selected baseline materialization — preview only

| Fact | Value |
|:---|:---|
| Selected match | seed 42, seat 0, 720 steps |
| Decision reason | cache file is missing |
| Cache filename | `seed-42-seat-0.json.gz` |
| Current lifecycle permits execution | No (`historical_reproduction`) |
| Baseline game run by this cell | No |

> This is a preview of the one selected action. Execution remains blocked until a separately approved experiment specification and explicit human decision authorize it.


### The first execution envelope: an approved smallest-matched smoke test

This is the important transition from *proving that we can refuse unsafe work* to *preparing the first narrowly authorized piece of real work*. The historical coarse experiment remains immutable evidence: it tells us what happened over 60 matched games, but it cannot grant itself permission to run again. Instead, the new approved contract below is a separate, source-controlled envelope for the smallest infrastructure-focused baseline suite that still preserves the system's matched-seat control.

Its scope is intentionally small: the pinned Kaggriculture target, the baseline `seed_opportunity_value=20`, seed 42 in seats 0 and 1, and 720 steps. The adapter materializes one selected cache entry at a time, beginning with seat 0; the contract keeps both seats because a one-seat suite would silently weaken the matched-comparison invariant enforced by the runner. It does not evaluate any candidate values, claim an aggregate reward result, alter policy source, or expand into the other 58 historical matches. If this matched baseline pair is later executed and reopened successfully, we will have earned confidence in the first real cache-writing slice of the adapter—not yet in the whole experiment.

The code cell validates this new contract through the reusable runner gates and previews the one selected action. It performs no simulator import or game run. The final operational prerequisite is explicit: this environment needs `kaggle_environments` before the materializer can be called.

In [207]:
import importlib
import importlib.util

import evaluator.runner as runner_module


# Notebook kernels retain imported modules. Reload the local runner so this preflight
# exercises the reviewed source currently on disk rather than an earlier class definition.
runner_module = importlib.reload(runner_module)
LocalInnerLoopAdapter = runner_module.LocalInnerLoopAdapter


# Use the reusable source adapter here: this is a contract preflight, not another notebook mock.
smoke_spec_path = find_workspace_file(
    Path("carrot-tuning-system/experiments/approved/carrot_investment/seed_opportunity_baseline_smoke_v1.json")
)
smoke_adapter = LocalInnerLoopAdapter()

# Reading through the adapter also applies the checked-in schema rather than trusting JSON parsing alone.
approved_smoke_spec = smoke_adapter.read_and_validate_spec(smoke_spec_path)

# The target remains explicit and fingerprinted before the approved lifecycle can matter.
approved_smoke_binding = smoke_adapter.bind_and_validate_target(
    approved_smoke_spec, project_root.parent.parent / "kaggriculture"
)
approved_smoke_lifecycle = smoke_adapter.authorize_lifecycle(approved_smoke_spec)
approved_smoke_controls = smoke_adapter.validate_local_controls(
    approved_smoke_spec, approved_smoke_binding
)

# Planning is safe: it derives both matched cache locations without creating their directory or files.
approved_smoke_cache_plan = smoke_adapter.plan_baseline_caches(
    approved_smoke_spec,
    approved_smoke_binding,
    project_root.parent / "var" / "carrot-tuning-system",
)
approved_smoke_cache_state = smoke_adapter.inspect_baseline_cache_state(
    approved_smoke_cache_plan
)
# A present path is reusable only after the source reader validates it; no simulator runs here.
approved_smoke_read_results = tuple(
    smoke_adapter.read_and_validate_baseline_cache(
        entry, approved_smoke_spec, approved_smoke_binding
    )
    for entry in approved_smoke_cache_state.present_entries
)
approved_smoke_preparation = smoke_adapter.plan_baseline_cache_preparation(
    approved_smoke_cache_state, approved_smoke_read_results
)
# Keep the reviewed first action stable across reruns: cache state may later change
# its decision from materialize to reuse, but never changes this approved match identity.
approved_smoke_action = next(
    action
    for action in approved_smoke_preparation.actions
    if (action.entry.seed, action.entry.seat) == (42, 0)
)

# Check availability without importing the simulator package or starting a game.
simulator_dependency_available = importlib.util.find_spec("kaggle_environments") is not None

assert approved_smoke_lifecycle.is_authorized
assert approved_smoke_controls.is_valid
assert len(approved_smoke_cache_plan.entries) == 2
assert (approved_smoke_action.entry.seed, approved_smoke_action.entry.seat) == (42, 0)
# Do not call smoke_adapter.materialize_selected_baseline here.


In [208]:
def render_approved_smoke_preflight() -> None:
    """Presentation only: render already-computed approved smoke-test facts."""
    dependency_status = (
        "Available" if simulator_dependency_available else "**No — install required before execution**"
    )
    display(Markdown(
        f"""### Approved smallest-matched smoke-test preflight

| Gate or fact | Result |
|:---|:---|
| Contract lifecycle | **{approved_smoke_lifecycle.declared_status.title()}** |
| Target fingerprint | Passed |
| Local controls | {'Passed' if approved_smoke_controls.is_valid else 'Failed'} |
| Contract suite | seed {approved_smoke_spec['suite']['seeds'][0]}, seats {approved_smoke_spec['suite']['seats']}, {approved_smoke_spec['suite']['steps']} steps |
| First selected baseline | seed {approved_smoke_action.entry.seed}, seat {approved_smoke_action.entry.seat} |
| Cache plan after validation | {len(approved_smoke_preparation.reusable_actions)} reusable cache(s); {len(approved_smoke_preparation.materialize_actions)} cache(s) still to materialize |
| Simulator dependency available | {dependency_status} |
| Game run by this cell | No |

> This preflight authorizes only the declared smallest matched smoke test after the simulator dependency is available and the human explicitly chooses to execute it. It does not run candidates or mutate policy source.
"""
    ))


render_approved_smoke_preflight()


### Approved smallest-matched smoke-test preflight

| Gate or fact | Result |
|:---|:---|
| Contract lifecycle | **Approved** |
| Target fingerprint | Passed |
| Local controls | Passed |
| Contract suite | seed 42, seats [0, 1], 720 steps |
| First selected baseline | seed 42, seat 0 |
| Cache plan after validation | 2 reusable cache(s); 0 cache(s) still to materialize |
| Simulator dependency available | Available |
| Game run by this cell | No |

> This preflight authorizes only the declared smallest matched smoke test after the simulator dependency is available and the human explicitly chooses to execute it. It does not run candidates or mutate policy source.


### Execute and reopen the first approved baseline cache

This is the first real simulator side effect in the adapter workbench. The earlier gates have already established the approved contract, the pinned target fingerprint, the full local-control envelope, and the exact missing cache entry to create. This cell now materializes **only** the first selected action: baseline `seed_opportunity_value=20`, seed 42, seat 0, and 720 steps.

The write is deliberately narrow and fail-closed. If the planned path already exists, the cell does not overwrite it; it only reopens and validates it. After a fresh write, it immediately reads the gzip cache through the same reusable runner boundary used for future reuse. That proves the cache is present, decodable, structurally sound, and bound to this exact target and match. It does **not** run seat 1, evaluate a candidate, aggregate rewards, write an evidence packet, or alter either policy source checkout.

A successful result is cache-infrastructure evidence for this one baseline match—not a candidate result or a reproduction claim. The remaining matched seat is still a separate approved action and will be reviewed next.

In [209]:
# This action came from the approved preflight above; do not select a new seed, seat, or path here.
assert (approved_smoke_action.entry.seed, approved_smoke_action.entry.seat) == (42, 0)

# A rerun must never overwrite evidence. If the first cache already exists, reopen it instead.
if approved_smoke_action.entry.path.exists():
    smoke_execution_status = "Already present; left untouched and reopened"
    smoke_materialization_result = None
else:
    # A missing entry must still be the preflight's explicit materialize decision.
    assert approved_smoke_action.decision == "materialize"
    # This is the only simulator call in this section: one approved baseline match.
    smoke_materialization_result = smoke_adapter.materialize_selected_baseline(
        approved_smoke_action, approved_smoke_spec, approved_smoke_binding
    )
    smoke_execution_status = "Materialized in this cell"

# Reopen through the reusable reader rather than trusting that a successful write implies reusable evidence.
reopened_smoke_cache = smoke_adapter.read_and_validate_baseline_cache(
    approved_smoke_action.entry, approved_smoke_spec, approved_smoke_binding
)
assert reopened_smoke_cache.is_reusable_candidate
assert reopened_smoke_cache.payload_validation is not None


In [210]:
def render_first_baseline_execution() -> None:
    """Presentation only: render facts produced by the execution and reopen cell."""
    validation = reopened_smoke_cache.payload_validation
    assert validation is not None
    checkpoint_count = (
        smoke_materialization_result.checkpoint_count
        if smoke_materialization_result is not None
        else "existing cache; checkpoint count not re-derived here"
    )
    display(Markdown(
        f"""### First approved baseline cache — execution record

| Fact | Result |
|:---|:---|
| Selected match | seed {approved_smoke_action.entry.seed}, seat {approved_smoke_action.entry.seat}, {approved_smoke_spec['suite']['steps']} steps |
| Cache action | {smoke_execution_status} |
| Cache path | `{approved_smoke_action.entry.path}` |
| Reopened gzip cache | {'Passed' if reopened_smoke_cache.read_error is None else 'Failed'} |
| Structural provenance checks | {'Passed' if validation.is_valid else 'Failed'} ({len(validation.checks)} checks) |
| Recorded checkpoints | {checkpoint_count} |
| Decision reward | {smoke_materialization_result.outcome['decision_reward'] if smoke_materialization_result else 'recorded in cache; not re-derived here'} |
| Opponent reward | {smoke_materialization_result.outcome['opponent_reward'] if smoke_materialization_result else 'recorded in cache; not re-derived here'} |
| Candidate evaluation by this section | No |
| Policy source change by this section | No |

> This establishes one reusable baseline cache for the approved smoke contract. It is not aggregate reward evidence and does not establish a candidate comparison.
"""
    ))


render_first_baseline_execution()


### First approved baseline cache — execution record

| Fact | Result |
|:---|:---|
| Selected match | seed 42, seat 0, 720 steps |
| Cache action | Already present; left untouched and reopened |
| Cache path | `/home/experiments/frontier-week/foundry-trainer/var/carrot-tuning-system/baseline-caches/5b527c025a39ccbbe607fe963103e61ef30a991a5649f9da7205e1c1db284b1d/seed-42-seat-0.json.gz` |
| Reopened gzip cache | Passed |
| Structural provenance checks | Passed (5 checks) |
| Recorded checkpoints | existing cache; checkpoint count not re-derived here |
| Decision reward | recorded in cache; not re-derived here |
| Opponent reward | recorded in cache; not re-derived here |
| Candidate evaluation by this section | No |
| Policy source change by this section | No |

> This establishes one reusable baseline cache for the approved smoke contract. It is not aggregate reward evidence and does not establish a candidate comparison.


## 9.2 Begin bounded candidate evaluation

The two baseline caches answer a narrow infrastructure question: can this adapter create, reopen, and validate the fixed reference matches? Yes. Candidate evaluation asks a different question: what happens if we supply one approved alternative parameter value to the exact same match? Before we let the checkpoint evaluator see a value, we need a small gate that makes the candidate envelope executable rather than merely descriptive.

This first component accepts one numeric value only when it appears in the contract's declared candidate list, and returns a one-field in-memory override. For this smoke contract, the only choices are `10.0`, `40.0`, `60.0`, and `80.0` for `seed_opportunity_value`. A value such as `25.0` may be interesting later, but it is not authorized by this contract and must fail here before any replay or simulator call.

The cell below selects `10.0` only as the first approved input to inspect. It does not read a cache, create candidate parameters, replay actions, restore a suffix, run a clean audit, or change policy source. Those are separate components after this selection gate is reviewed.

In [211]:
from numbers import Real


def select_declared_candidate_override(
    specification: dict, candidate_value: Real
) -> dict[str, float]:
    """Return one approved in-memory override; perform no evaluation."""
    # Reject booleans explicitly: Python treats True as the number 1, but it is not a policy value.
    if isinstance(candidate_value, bool) or not isinstance(candidate_value, Real):
        raise ValueError("candidate value must be a real number")

    parameter = specification["parameter"]
    declared_values = tuple(parameter["candidate_values"])
    resolved_value = float(candidate_value)

    # Fail before touching a cache or simulator if the contract did not name this exact value.
    if resolved_value not in declared_values:
        raise ValueError(
            f"candidate {resolved_value} is not declared; allowed values: {declared_values}"
        )

    # One mapping expresses the only permitted future policy input; it is not a source mutation.
    return {parameter["name"]: resolved_value}


first_candidate_override = select_declared_candidate_override(
    approved_smoke_spec, 10.0
)

# Focused counterexample: an undeclared value is rejected before evaluation can begin.
try:
    select_declared_candidate_override(approved_smoke_spec, 25.0)
except ValueError as error:
    undeclared_candidate_error = str(error)
else:
    raise AssertionError("an undeclared candidate must not pass the selection gate")

assert first_candidate_override == {"seed_opportunity_value": 10.0}


In [212]:
def render_declared_candidate_selection() -> None:
    """Presentation only: render already-selected override and rejection facts."""
    parameter_name, selected_value = next(iter(first_candidate_override.items()))
    allowed_values = ", ".join(
        f"`{value}`" for value in approved_smoke_spec["parameter"]["candidate_values"]
    )
    display(Markdown(
        f"""### Declared candidate selection

| Fact | Result |
|:---|:---|
| Declared parameter | `{parameter_name}` |
| Allowed values | {allowed_values} |
| Selected first override | `{parameter_name}={selected_value}` |
| Undeclared `25.0` | Rejected before evaluation (`{undeclared_candidate_error}`) |
| Cache read or game run by this section | No |
| Policy source change by this section | No |

> This is an input-authorization gate, not candidate evidence. The next component will apply this already-approved override to one validated cache.
"""
    ))


render_declared_candidate_selection()


### Declared candidate selection

| Fact | Result |
|:---|:---|
| Declared parameter | `seed_opportunity_value` |
| Allowed values | `10.0`, `40.0`, `60.0`, `80.0` |
| Selected first override | `seed_opportunity_value=10.0` |
| Undeclared `25.0` | Rejected before evaluation (`candidate 25.0 is not declared; allowed values: (10.0, 40.0, 60.0, 80.0)`) |
| Cache read or game run by this section | No |
| Policy source change by this section | No |

> This is an input-authorization gate, not candidate evidence. The next component will apply this already-approved override to one validated cache.


### Replay one declared candidate from one validated baseline cache

The selection gate has authorized the single in-memory override `seed_opportunity_value=10.0`. This component applies that override to one already validated reference match: seed 42, seat 0. It first reuses the runner's cache reader and provenance validator; only then does it hand the decoded cache to the proven checkpoint evaluator.

The checkpoint evaluator replays the baseline observations and compares the candidate's actions with the stored baseline actions. If every action agrees, it returns the baseline outcome as a fast `reused-baseline` result. If an action differs, it restores the exact pre-action checkpoint and simulates only the remaining suffix. Either result is still **unaudited** at this point. A clean full candidate rerun is the next component and remains the oracle.

This cell runs only value `10.0`, seed 42, seat 0. It does not evaluate seat 1 or the other declared values, aggregate a reward claim, write an evidence packet, or modify policy source.

In [213]:
from kaggriculture_eval.checkpoint import evaluate_cached, read_baseline


def evaluate_declared_candidate_from_cache(
    override: dict[str, float],
    entry,
) -> dict:
    """Return one fast, explicitly unaudited candidate result from a validated cache."""
    # Reapply the declared-value gate so this wrapper cannot be called with an ad hoc override.
    if len(override) != 1:
        raise ValueError("a candidate override must contain exactly one field")
    name, value = next(iter(override.items()))
    if select_declared_candidate_override(approved_smoke_spec, value) != override:
        raise ValueError("candidate override does not match the approved parameter")

    # A path is not evidence by itself. Read through the runner and fail closed unless provenance passes.
    cache_read = smoke_adapter.read_and_validate_baseline_cache(
        entry, approved_smoke_spec, approved_smoke_binding
    )
    if not cache_read.is_reusable_candidate:
        raise ValueError("candidate evaluation requires a reusable validated baseline cache")

    cache = read_baseline(entry.path)
    result = evaluate_cached(
        approved_smoke_binding,
        cache,
        override,
        opponent=approved_smoke_binding.shared.carrot_conveyor_agent,
    )

    # The fast evaluator must not accidentally be presented as an audit.
    assert result["audit_status"] == "not-run"
    return result


seat_zero_entry = next(
    entry
    for entry in approved_smoke_cache_plan.entries
    if (entry.seed, entry.seat) == (42, 0)
)
fast_candidate_result = evaluate_declared_candidate_from_cache(
    first_candidate_override, seat_zero_entry
)

assert fast_candidate_result["candidate"] == {
    "name": "seed_opportunity_value",
    "old_value": 20.0,
    "new_value": 10.0,
}


In [214]:
def render_fast_candidate_result() -> None:
    """Presentation only: render facts already returned by the fast evaluator."""
    outcome = fast_candidate_result["outcome"]
    divergence = fast_candidate_result["first_divergence_turn"]
    display(Markdown(
        f"""### First candidate fast result — audit pending

| Fact | Result |
|:---|:---|
| Candidate override | `{fast_candidate_result['candidate']['name']}={fast_candidate_result['candidate']['new_value']}` |
| Matched baseline | seed {seat_zero_entry.seed}, seat {seat_zero_entry.seat}, {approved_smoke_spec['suite']['steps']} steps |
| Fast-path mode | `{fast_candidate_result['mode']}` |
| First action divergence | {divergence if divergence is not None else 'None'} |
| Simulated suffix actions | {fast_candidate_result['suffix_actions']} |
| Fast decision reward | {outcome['decision_reward']} |
| Fast opponent reward | {outcome['opponent_reward']} |
| Clean audit | **Not run** |
| Aggregate or candidate conclusion | None |

> This is one fast checkpoint/replay result. It becomes usable evidence only if the next clean full-run audit matches it.
"""
    ))


render_fast_candidate_result()


### First candidate fast result — audit pending

| Fact | Result |
|:---|:---|
| Candidate override | `seed_opportunity_value=10.0` |
| Matched baseline | seed 42, seat 0, 720 steps |
| Fast-path mode | `reused-baseline` |
| First action divergence | None |
| Simulated suffix actions | 0 |
| Fast decision reward | 8849.0 |
| Fast opponent reward | 8729.0 |
| Clean audit | **Not run** |
| Aggregate or candidate conclusion | None |

> This is one fast checkpoint/replay result. It becomes usable evidence only if the next clean full-run audit matches it.


### Clean-audit the first fast result

A fast checkpoint/replay result is efficient, but the clean audit is what makes it credible. This component starts a brand-new candidate game from turn zero using the same pinned target, seed 42, seat 0, 720-step limit, deterministic opponent, and already-declared `seed_opportunity_value=10.0` override. It does not read checkpoints to decide the candidate's actions.

After the clean run finishes, the proven audit helper compares only the final candidate outcome with the fast result's outcome. A mismatch is a hard stop: the fast result is unusable, and we do not proceed to seat 1 or another candidate. A match audits this one candidate/match pair only; it is not a two-seat result, an aggregate finding, or a general recommendation.

This is the first independently simulated candidate match in the workbench. It remains an in-memory parameter override and does not write policy source, alter the experiment envelope, or write an evidence packet.

In [215]:
from copy import deepcopy

from kaggriculture_eval.checkpoint import audit_cached


def audit_one_fast_candidate_result(
    fast_result: dict, override: dict[str, float]
) -> dict:
    """Run one clean oracle and return its audited comparison; no aggregation."""
    # A clean audit must correspond to a known, still-unaudited fast result.
    if fast_result.get("audit_status") != "not-run":
        raise ValueError("a clean audit requires an unaudited fast result")
    if select_declared_candidate_override(approved_smoke_spec, next(iter(override.values()))) != override:
        raise ValueError("clean audit override does not match the declared candidate")

    match = fast_result["match"]
    if (match["seed"], match["decision_player"]) != (42, 0):
        raise ValueError("this reviewed cell audits only seed 42, seat 0")

    # audit_cached performs a full candidate run from turn zero and marks pass/fail by outcome equality.
    return audit_cached(
        approved_smoke_binding,
        deepcopy(fast_result),
        override,
        opponent=approved_smoke_binding.shared.carrot_conveyor_agent,
    )


audited_candidate_result = audit_one_fast_candidate_result(
    fast_candidate_result, first_candidate_override
)

assert audited_candidate_result["audit_status"] == "passed"
assert audited_candidate_result["clean_audit"]["outcome"] == audited_candidate_result["outcome"]


In [216]:
def render_first_candidate_audit() -> None:
    """Presentation only: render the already-computed fast/clean audit comparison."""
    clean_audit = audited_candidate_result["clean_audit"]
    fast_outcome = audited_candidate_result["outcome"]
    clean_outcome = clean_audit["outcome"]
    display(Markdown(
        f"""### First candidate clean audit — passed

| Fact | Result |
|:---|:---|
| Candidate override | `seed_opportunity_value=10.0` |
| Matched game | seed 42, seat 0, {approved_smoke_spec['suite']['steps']} steps |
| Fast-path mode | `{audited_candidate_result['mode']}` |
| Clean-run actions | {clean_audit['actions_run']} |
| Fast decision/opponent reward | {fast_outcome['decision_reward']} / {fast_outcome['opponent_reward']} |
| Clean decision/opponent reward | {clean_outcome['decision_reward']} / {clean_outcome['opponent_reward']} |
| Audit comparison | **{audited_candidate_result['audit_status'].title()}** |
| Aggregate or two-seat conclusion | None |

> This one fast result matches its independent clean rerun. Seat 1 and every other candidate still require their own fast result and clean audit.
"""
    ))


render_first_candidate_audit()


### First candidate clean audit — passed

| Fact | Result |
|:---|:---|
| Candidate override | `seed_opportunity_value=10.0` |
| Matched game | seed 42, seat 0, 720 steps |
| Fast-path mode | `reused-baseline` |
| Clean-run actions | 719 |
| Fast decision/opponent reward | 8849.0 / 8729.0 |
| Clean decision/opponent reward | 8849.0 / 8729.0 |
| Audit comparison | **Passed** |
| Aggregate or two-seat conclusion | None |

> This one fast result matches its independent clean rerun. Seat 1 and every other candidate still require their own fast result and clean audit.


### Complete the remaining approved candidate audits as one bounded batch

The first `10.0` / seat-0 pair has already passed its clean audit. The remaining work is seven independent candidate/match pairs: `10.0` in seat 1, then `40.0`, `60.0`, and `80.0` in both seats. This single cell runs each pair in the same fixed order: validate the already-created baseline cache, obtain the fast checkpoint/replay result, immediately run its clean full-game audit, and stop at the first failure.

The batch is bounded by the approved contract, not by a loop that invents work. It cannot add a value, seed, seat, opponent, or source edit. Its output is a per-match audit record, not an aggregate, a ranking, or a policy recommendation. Aggregation is deliberately a later component.

In [217]:
# This list is the complete remainder after the separately audited 10.0 / seat-0 pair.
remaining_candidate_tasks = (
    (10.0, 1),
    (40.0, 0),
    (40.0, 1),
    (60.0, 0),
    (60.0, 1),
    (80.0, 0),
    (80.0, 1),
)


def run_remaining_approved_candidate_audits() -> tuple[dict, ...]:
    """Run only the declared remaining tasks and fail closed on the first audit mismatch."""
    declared_values = tuple(approved_smoke_spec["parameter"]["candidate_values"])
    declared_seats = tuple(approved_smoke_spec["suite"]["seats"])
    expected_tasks = {(value, seat) for value in declared_values for seat in declared_seats}
    completed_task = (10.0, 0)

    # The literal batch list must be exactly the approved cross-product minus the one audited task.
    if set(remaining_candidate_tasks) != expected_tasks - {completed_task}:
        raise AssertionError("batch tasks differ from the approved remaining candidate envelope")

    results = []
    for candidate_value, seat in remaining_candidate_tasks:
        override = select_declared_candidate_override(approved_smoke_spec, candidate_value)
        entry = next(
            planned_entry
            for planned_entry in approved_smoke_cache_plan.entries
            if (planned_entry.seed, planned_entry.seat) == (42, seat)
        )

        # Reuse requires a fresh provenance/structure check for the exact planned match.
        cache_read = smoke_adapter.read_and_validate_baseline_cache(
            entry, approved_smoke_spec, approved_smoke_binding
        )
        if not cache_read.is_reusable_candidate:
            raise ValueError(f"task {(candidate_value, seat)} lacks a reusable validated baseline cache")

        cache = read_baseline(entry.path)
        fast = evaluate_cached(
            approved_smoke_binding,
            cache,
            override,
            opponent=approved_smoke_binding.shared.carrot_conveyor_agent,
        )
        audited = audit_cached(
            approved_smoke_binding,
            fast,
            override,
            opponent=approved_smoke_binding.shared.carrot_conveyor_agent,
        )

        # A failed audit is not a partial success: stop before the next approved task.
        if audited["audit_status"] != "passed":
            raise RuntimeError(f"clean audit failed for {(candidate_value, seat)}: {audited}")
        results.append(audited)

    return tuple(results)


remaining_audited_candidate_results = run_remaining_approved_candidate_audits()
assert len(remaining_audited_candidate_results) == len(remaining_candidate_tasks)
assert all(result["audit_status"] == "passed" for result in remaining_audited_candidate_results)


In [218]:
def render_remaining_candidate_audits() -> None:
    """Presentation only: render already-audited per-match facts without aggregating them."""
    rows = "\n".join(
        f"| {result['candidate']['new_value']} | {result['match']['decision_player']} | "
        f"`{result['mode']}` | "
        f"{result['first_divergence_turn'] if result['first_divergence_turn'] is not None else 'None'} | "
        f"{result['suffix_actions']} | "
        f"{result['outcome']['decision_reward']} / {result['outcome']['opponent_reward']} | "
        f"**{result['audit_status'].title()}** |"
        for result in remaining_audited_candidate_results
    )
    display(Markdown(
        f"""### Remaining approved candidate audits — per-match record

| Value | Seat | Fast mode | First divergence | Suffix actions | Audited decision/opponent reward | Audit |
|---:|---:|:---|---:|---:|:---|:---|
{rows}

> Every row is a separate fast-result/clean-audit pair. This table deliberately does not compute a mean, rank a value, or recommend a policy change.
"""
    ))


render_remaining_candidate_audits()


### Remaining approved candidate audits — per-match record

| Value | Seat | Fast mode | First divergence | Suffix actions | Audited decision/opponent reward | Audit |
|---:|---:|:---|---:|---:|:---|:---|
| 10.0 | 1 | `reused-baseline` | None | 0 | 8809.0 / 8789.0 | **Passed** |
| 40.0 | 0 | `reused-baseline` | None | 0 | 8849.0 / 8729.0 | **Passed** |
| 40.0 | 1 | `reused-baseline` | None | 0 | 8809.0 / 8789.0 | **Passed** |
| 60.0 | 0 | `reused-baseline` | None | 0 | 8849.0 / 8729.0 | **Passed** |
| 60.0 | 1 | `reused-baseline` | None | 0 | 8809.0 / 8789.0 | **Passed** |
| 80.0 | 0 | `restored-suffix` | 107 | 612 | 9776.0 / 10846.0 | **Passed** |
| 80.0 | 1 | `restored-suffix` | 107 | 612 | 9756.0 / 10886.0 | **Passed** |

> Every row is a separate fast-result/clean-audit pair. This table deliberately does not compute a mean, rank a value, or recommend a policy change.


## 9.3 Build the local evidence packet

The eight fast-result/clean-audit pairs are now real local facts. Before anything interprets them, we need to put those facts into one explicit record whose shape can be checked. This first packet is **in memory only**: it neither writes a file nor calls a simulator. Its job is to make provenance, each audited match, timing components, and the paired smoke aggregates inspectable together.

The builder below is deliberately strict. It rejects a missing, duplicated, undeclared, unaudited, or mismatched result before producing a packet. It reopens the already validated baseline caches only to obtain the matched baseline outcomes used for paired deltas; it does not create or overwrite a cache. The aggregate section is descriptive for this one-seed smoke suite. In particular, it is not a candidate ranking, it does not replace the 60-match historical conclusion, and it cannot authorize a policy change or the next experiment.


In [220]:
from dataclasses import asdict, dataclass
from typing import Any

from kaggriculture_eval.checkpoint import read_baseline


@dataclass(frozen=True)
class EvidenceMatchRecord:
    """One fully audited candidate/match fact retained in the local packet."""

    candidate_value: float
    seed: int
    seat: int
    fast_mode: str
    first_divergence_turn: int | None
    suffix_actions: int
    baseline_decision_reward: float
    candidate_decision_reward: float
    paired_decision_delta: float
    opponent_reward: float
    replay_elapsed_seconds: float
    suffix_elapsed_seconds: float
    clean_actions_run: int
    clean_elapsed_seconds: float
    audit_status: str


@dataclass(frozen=True)
class EvidenceAggregate:
    """Descriptive paired summary; deliberately not an interpretation or ranking."""

    candidate_value: float
    match_count: int
    all_audits_passed: bool
    mean_candidate_decision_reward: float
    mean_paired_decision_delta: float
    divergent_match_count: int
    total_suffix_actions: int


@dataclass(frozen=True)
class LocalSmokeEvidencePacket:
    """Local, serializable evidence shape for one already-executed smoke envelope."""

    packet_schema_version: int
    experiment_id: str
    target_revision: str
    target_shared_sha256: str
    target_engine_sha256: str
    parameter_name: str
    baseline_value: float
    suite_seeds: tuple[int, ...]
    suite_seats: tuple[int, ...]
    suite_steps: int
    per_match: tuple[EvidenceMatchRecord, ...]
    aggregates: tuple[EvidenceAggregate, ...]
    all_audits_passed: bool
    interpretation_status: str
    source_policy_mutation_authorized: bool

    def as_jsonable(self) -> dict[str, Any]:
        """Return data ready for a later local JSON writer; do not write it here."""
        return asdict(self)


def read_validated_smoke_baseline_outcomes() -> dict[tuple[int, int], dict[str, Any]]:
    """Read only already-validated baseline outcomes needed for paired deltas."""
    outcomes: dict[tuple[int, int], dict[str, Any]] = {}
    for entry in approved_smoke_cache_plan.entries:
        # Pairing must start from the same target-bound cache that evaluation reused.
        validated = smoke_adapter.read_and_validate_baseline_cache(
            entry, approved_smoke_spec, approved_smoke_binding
        )
        if not validated.is_reusable_candidate:
            raise ValueError(f"baseline cache is not reusable for {(entry.seed, entry.seat)}")
        outcomes[(entry.seed, entry.seat)] = read_baseline(entry.path)["outcome"]
    return outcomes


def build_local_smoke_evidence_packet(
    specification: dict[str, Any],
    binding: Any,
    audited_results: tuple[dict[str, Any], ...],
    baseline_outcomes: dict[tuple[int, int], dict[str, Any]],
) -> LocalSmokeEvidencePacket:
    """Validate complete audited coverage and assemble descriptive local evidence."""
    parameter = specification["parameter"]
    suite = specification["suite"]
    declared_values = tuple(float(value) for value in parameter["candidate_values"])
    expected_matches = {
        (candidate_value, seed, seat)
        for candidate_value in declared_values
        for seed in suite["seeds"]
        for seat in suite["seats"]
    }

    observed_matches = {
        (
            float(result["candidate"]["new_value"]),
            result["match"]["seed"],
            result["match"]["decision_player"],
        )
        for result in audited_results
    }
    # Exact set equality catches missing, duplicate, and out-of-envelope results.
    if len(audited_results) != len(observed_matches) or observed_matches != expected_matches:
        raise ValueError("audited results do not exactly cover the approved smoke envelope")

    records = []
    for result in audited_results:
        candidate = result["candidate"]
        match = result["match"]
        clean_audit = result.get("clean_audit")
        match_key = (match["seed"], match["decision_player"])

        # Require the independent oracle before a result may enter durable evidence.
        if result.get("audit_status") != "passed" or clean_audit is None:
            raise ValueError(f"result {match_key} is not a passing clean audit")
        if clean_audit["outcome"] != result["outcome"]:
            raise ValueError(f"fast and clean outcomes differ for {match_key}")
        if candidate["name"] != parameter["name"] or candidate["old_value"] != parameter["baseline_value"]:
            raise ValueError(f"result {match_key} does not use the declared parameter baseline")
        if result["candidate_source_revision"] != binding.revision:
            raise ValueError(f"candidate target revision differs for {match_key}")
        if match_key not in baseline_outcomes:
            raise ValueError(f"missing baseline outcome for {match_key}")

        is_reused = result["mode"] == "reused-baseline"
        is_suffix = result["mode"] == "restored-suffix"
        if not (is_reused or is_suffix):
            raise ValueError(f"unknown fast-path mode for {match_key}: {result['mode']}")
        if is_reused and (result["first_divergence_turn"] is not None or result["suffix_actions"] != 0):
            raise ValueError(f"reused baseline has inconsistent divergence facts for {match_key}")
        if is_suffix and (result["first_divergence_turn"] is None or result["suffix_actions"] <= 0):
            raise ValueError(f"restored suffix has inconsistent divergence facts for {match_key}")

        baseline = baseline_outcomes[match_key]
        outcome = result["outcome"]
        records.append(EvidenceMatchRecord(
            candidate_value=float(candidate["new_value"]),
            seed=match["seed"],
            seat=match["decision_player"],
            fast_mode=result["mode"],
            first_divergence_turn=result["first_divergence_turn"],
            suffix_actions=result["suffix_actions"],
            baseline_decision_reward=float(baseline["decision_reward"]),
            candidate_decision_reward=float(outcome["decision_reward"]),
            paired_decision_delta=float(outcome["decision_reward"] - baseline["decision_reward"]),
            opponent_reward=float(outcome["opponent_reward"]),
            replay_elapsed_seconds=float(result["replay_elapsed_seconds"]),
            suffix_elapsed_seconds=float(result["suffix_elapsed_seconds"]),
            clean_actions_run=clean_audit["actions_run"],
            clean_elapsed_seconds=float(clean_audit["elapsed_seconds"]),
            audit_status=result["audit_status"],
        ))

    ordered_records = tuple(sorted(records, key=lambda record: (
        declared_values.index(record.candidate_value), record.seed, record.seat
    )))
    aggregates = tuple(
        EvidenceAggregate(
            candidate_value=value,
            match_count=len(candidate_records := tuple(
                record for record in ordered_records if record.candidate_value == value
            )),
            all_audits_passed=all(record.audit_status == "passed" for record in candidate_records),
            mean_candidate_decision_reward=sum(
                record.candidate_decision_reward for record in candidate_records
            ) / len(candidate_records),
            mean_paired_decision_delta=sum(
                record.paired_decision_delta for record in candidate_records
            ) / len(candidate_records),
            divergent_match_count=sum(
                record.first_divergence_turn is not None for record in candidate_records
            ),
            total_suffix_actions=sum(record.suffix_actions for record in candidate_records),
        )
        for value in declared_values
    )

    return LocalSmokeEvidencePacket(
        packet_schema_version=1,
        experiment_id=specification["experiment_id"],
        target_revision=binding.revision,
        target_shared_sha256=binding.shared_sha256,
        target_engine_sha256=binding.engine_sha256,
        parameter_name=parameter["name"],
        baseline_value=float(parameter["baseline_value"]),
        suite_seeds=tuple(suite["seeds"]),
        suite_seats=tuple(suite["seats"]),
        suite_steps=suite["steps"],
        per_match=ordered_records,
        aggregates=aggregates,
        all_audits_passed=all(record.audit_status == "passed" for record in ordered_records),
        interpretation_status="descriptive-smoke-only; human interpretation required",
        source_policy_mutation_authorized=specification["governance"][
            "source_policy_mutation_authorized"
        ],
    )


# This combines the first detailed example with the fixed seven-pair batch; it never runs a game.
all_audited_smoke_results = (audited_candidate_result, *remaining_audited_candidate_results)
smoke_baseline_outcomes = read_validated_smoke_baseline_outcomes()
local_smoke_evidence_packet = build_local_smoke_evidence_packet(
    approved_smoke_spec,
    approved_smoke_binding,
    all_audited_smoke_results,
    smoke_baseline_outcomes,
)

assert len(local_smoke_evidence_packet.per_match) == 8
assert local_smoke_evidence_packet.all_audits_passed
assert all(aggregate.match_count == 2 for aggregate in local_smoke_evidence_packet.aggregates)
assert not local_smoke_evidence_packet.source_policy_mutation_authorized


In [221]:
def render_local_smoke_evidence_packet(packet: LocalSmokeEvidencePacket) -> None:
    """Presentation only: show existing packet facts without interpreting or writing them."""
    aggregate_rows = "\n".join(
        f"| {aggregate.candidate_value:.1f} | {aggregate.match_count} | "
        f"{aggregate.mean_candidate_decision_reward:.1f} | "
        f"{aggregate.mean_paired_decision_delta:+.1f} | "
        f"{aggregate.divergent_match_count} | {aggregate.total_suffix_actions} | "
        f"**{'Passed' if aggregate.all_audits_passed else 'Failed'}** |"
        for aggregate in packet.aggregates
    )
    display(Markdown(
        f"""### Local smoke evidence packet — descriptive only

| Packet fact | Value |
|:---|:---|
| Packet schema | `local-smoke-evidence-v{packet.packet_schema_version}` |
| Experiment | `{packet.experiment_id}` |
| Pinned target revision | `{packet.target_revision}` |
| Parameter / baseline | `{packet.parameter_name}={packet.baseline_value:.1f}` |
| Matched smoke suite | seeds {packet.suite_seeds}; seats {packet.suite_seats}; {packet.suite_steps} steps |
| Audited per-match records | {len(packet.per_match)} |
| All clean audits | **{'Passed' if packet.all_audits_passed else 'Failed'}** |
| Policy-source mutation authorized | **No** |

### Descriptive paired smoke aggregates

| Value | Matches | Mean candidate reward | Mean paired delta | Divergent matches | Suffix actions | Audits |
|---:|---:|---:|---:|---:|---:|:---|
{aggregate_rows}

> These aggregates summarize one seed in two seats. They are retained so a later reviewer can inspect coverage and paired deltas, but they are **not** a ranking, a replacement for the 60-match historical result, a policy recommendation, or authorization to run another experiment. The model cell retains per-match fast-path, timing, clean-audit, and provenance facts; this renderer only summarizes them.
"""
    ))


render_local_smoke_evidence_packet(local_smoke_evidence_packet)


### Local smoke evidence packet — descriptive only

| Packet fact | Value |
|:---|:---|
| Packet schema | `local-smoke-evidence-v1` |
| Experiment | `carrot-seed-opportunity-baseline-smoke-v1` |
| Pinned target revision | `6ace5fed9018fef45642b41ea11d0bd741a508a4` |
| Parameter / baseline | `seed_opportunity_value=20.0` |
| Matched smoke suite | seeds (42,); seats (0, 1); 720 steps |
| Audited per-match records | 8 |
| All clean audits | **Passed** |
| Policy-source mutation authorized | **No** |

### Descriptive paired smoke aggregates

| Value | Matches | Mean candidate reward | Mean paired delta | Divergent matches | Suffix actions | Audits |
|---:|---:|---:|---:|---:|---:|:---|
| 10.0 | 2 | 8829.0 | +0.0 | 0 | 0 | **Passed** |
| 40.0 | 2 | 8829.0 | +0.0 | 0 | 0 | **Passed** |
| 60.0 | 2 | 8829.0 | +0.0 | 0 | 0 | **Passed** |
| 80.0 | 2 | 9766.0 | +937.0 | 2 | 1224 | **Passed** |

> These aggregates summarize one seed in two seats. They are retained so a later reviewer can inspect coverage and paired deltas, but they are **not** a ranking, a replacement for the 60-match historical result, a policy recommendation, or authorization to run another experiment. The model cell retains per-match fast-path, timing, clean-audit, and provenance facts; this renderer only summarizes them.


### Promote the reviewed packet and exercise its writer safely

The packet model and its exact-coverage checks have now moved into the reusable adapter. The source writer accepts only a complete packet whose clean audits all passed, writes JSON under a caller-provided local report directory, and refuses to overwrite an existing packet with the same experiment identifier. It does not run Kaggriculture or edit policy source.

The exercise below deliberately uses a temporary directory. It builds the source packet from the already-audited notebook values, writes it once, reopens the JSON, and confirms that a second write fails closed. The temporary directory is removed automatically at the end of the cell, so this verifies writer behavior without creating the first durable smoke evidence artifact under `var/` yet. A later reviewed orchestration step will choose when the real runner is allowed to write that durable packet.


In [ ]:
from tempfile import TemporaryDirectory


# Reload so the notebook exercises the reviewed reusable source rather than a prior kernel import.
runner_module = importlib.reload(runner_module)

# The source builder must preserve the same packet facts that the notebook prototype produced.
source_smoke_evidence_packet = runner_module.build_local_smoke_evidence_packet(
    approved_smoke_spec,
    approved_smoke_binding,
    all_audited_smoke_results,
    smoke_baseline_outcomes,
)
assert source_smoke_evidence_packet.as_jsonable() == local_smoke_evidence_packet.as_jsonable()

with TemporaryDirectory() as temporary_report_directory:
    temporary_report_root = Path(temporary_report_directory)
    temporary_write_result = runner_module.write_local_smoke_evidence_packet(
        source_smoke_evidence_packet, temporary_report_root
    )
    # Reopening proves the writer emitted JSON for the exact reviewed packet.
    reopened_packet = json.loads(temporary_write_result.path.read_text(encoding="utf-8"))
    expected_json_packet = json.loads(json.dumps(source_smoke_evidence_packet.as_jsonable()))
    assert reopened_packet == expected_json_packet

    try:
        # A repeat would erase the boundary between one immutable artifact and a rewrite.
        runner_module.write_local_smoke_evidence_packet(
            source_smoke_evidence_packet, temporary_report_root
        )
    except FileExistsError:
        duplicate_write_refused = True
    else:
        duplicate_write_refused = False

assert duplicate_write_refused
assert not temporary_report_root.exists()
source_writer_exercise = {
    "source_packet_matches_notebook": True,
    "json_reopen_passed": True,
    "duplicate_write_refused": duplicate_write_refused,
    "temporary_artifact_removed": True,
    "simulator_run": False,
}


In [ ]:
def render_source_packet_writer_exercise() -> None:
    """Presentation only: render existing temporary-fixture writer facts."""
    display(Markdown(
        f"""### Reusable evidence writer — temporary-fixture exercise

| Check | Result |
|:---|:---|
| Source packet matches reviewed notebook packet | **Passed** |
| One temporary JSON write and reopen | **Passed** |
| Second write to the same destination | **Refused** |
| Temporary artifact after the cell | Removed |
| Simulator run | No |
| Policy-source change | No |

> This proves the reviewed source boundary and its overwrite guard. It does not create the durable packet for the smoke run or make the one-seed aggregate interpretable as a recommendation.
"""
    ))


render_source_packet_writer_exercise()


### Reusable evidence writer — temporary-fixture exercise

| Check | Result |
|:---|:---|
| Source packet matches reviewed notebook packet | **Passed** |
| One temporary JSON write and reopen | **Passed** |
| Second write to the same destination | **Refused** |
| Temporary artifact after the cell | Removed |
| Simulator run | No |
| Policy-source change | No |

> This proves the reviewed source boundary and its overwrite guard. It does not create the durable packet for the smoke run or make the one-seed aggregate interpretable as a recommendation.


### Promote bounded candidate evaluation and audit helpers

The reusable adapter now owns the same three boundaries reviewed above: selecting a declared one-field override, obtaining one fast result only from an exact validated cache, and requiring that result's matching clean audit. The combined helper performs exactly one approved candidate/match pair and stops with an error if the audit disagrees. It does not choose another value, continue after a failure, aggregate results, write an evidence packet, or alter policy source.

The source check below exercises only guards that finish before a simulator could start: an undeclared value is rejected before cache access, and an already-audited result is rejected before a duplicate clean rerun. The already-recorded eight audited matches remain the actual game evidence; this promotion exercise intentionally does not rerun them.


In [222]:
# Reload so this exercise calls the promoted reusable boundaries, not notebook-local prototypes.
runner_module = importlib.reload(runner_module)

source_declared_override = runner_module.select_declared_candidate_override(
    approved_smoke_spec, 10.0
)
assert source_declared_override == first_candidate_override

try:
    runner_module.select_declared_candidate_override(approved_smoke_spec, 25.0)
except ValueError:
    source_undeclared_candidate_refused = True
else:
    source_undeclared_candidate_refused = False

# This synthetic status is deliberately rejected before the source audit helper imports or runs a simulator.
try:
    runner_module.audit_declared_candidate_result(
        approved_smoke_spec,
        approved_smoke_binding,
        {"audit_status": "passed"},
        source_declared_override,
    )
except ValueError:
    source_duplicate_audit_refused = True
else:
    source_duplicate_audit_refused = False

assert source_undeclared_candidate_refused
assert source_duplicate_audit_refused
source_candidate_promotion_exercise = {
    "declared_override_matches_notebook": True,
    "undeclared_value_refused_before_cache": source_undeclared_candidate_refused,
    "duplicate_audit_refused_before_simulation": source_duplicate_audit_refused,
    "simulator_run": False,
}


In [223]:
def render_source_candidate_promotion_exercise() -> None:
    """Presentation only: render existing source-boundary guard facts."""
    display(Markdown(
        """### Reusable candidate helpers — guard exercise

| Check | Result |
|:---|:---|
| Declared `10.0` override matches notebook gate | **Passed** |
| Undeclared `25.0` rejected before cache access | **Passed** |
| Already-audited result rejected before duplicate clean run | **Passed** |
| Simulator run by this exercise | No |
| New evidence artifact | No |

> The source helper is ready to be called by a later reviewed `run()` stage. This exercise verifies its stop conditions only; it does not replace the eight already-recorded audited matches.
"""
    ))


render_source_candidate_promotion_exercise()


### Reusable candidate helpers — guard exercise

| Check | Result |
|:---|:---|
| Declared `10.0` override matches notebook gate | **Passed** |
| Undeclared `25.0` rejected before cache access | **Passed** |
| Already-audited result rejected before duplicate clean run | **Passed** |
| Simulator run by this exercise | No |
| New evidence artifact | No |

> The source helper is ready to be called by a later reviewed `run()` stage. This exercise verifies its stop conditions only; it does not replace the eight already-recorded audited matches.


## 9.4 Verify reproduction before adding intelligence

Not started. The first real run must reproduce the historical classifications: 10 inert at scale; 40, 60, and 80 directional harm; all required clean audits passing. A mismatch is a runner repair task, not an invitation to alter the candidate envelope or explain the result away.

## 9.5 Preserve the broader roadmap

This notebook stays focused on the local adapter. Only after the reproduction smoke test succeeds do we move to a graph-adapter contract, then deterministic outer evaluation, and finally a Foundry-facing reasoning adapter. None of those later components score the game or edit policy source.

## Completion and required pre-real-system gate

The reusable adapter now has reviewed one-entry baseline materialization, bounded candidate evaluation with mandatory clean audit, and immutable evidence writing. This workbench is complete.

### Pre-real-system regression gate — complete

Before a completed component workbench may be wired into a real orchestrator or an approved end-to-end notebook, its promoted source boundaries need a fast, simulator-free regression suite. The suite is derived from the accepted notebook success and guard cases, tests the source module rather than notebook renderers, and includes relevant failure edges. Here, `tests/test_local_inner_loop_adapter.py` covers declared-value and suite rejection before cache access, cache validation, duplicate/mismatched audit refusal, exact evidence coverage, malformed input, packet round-trip/safety, and overwrite refusal. The default run passed 19 tests and deselected the separately marked real-simulator regression.

This gate preserves reviewed behavior during integration; it does not create game evidence, approve a candidate envelope, replace a clean audit, or authorize a simulator run.

### Next reviewed change

The current approved smoke contract says it does not evaluate candidates, so the next slice needs a new candidate-evidence contract and a separate end-to-end-run notebook. Do not use prior notebook work to silently broaden that contract, add candidates, expand the suite, interpret the one-seed aggregate as a ranking, or alter policy source.
